# MITgcm preprocessing
Builds a ready-to-compile MITgcm run folder in `LZT_datafiles/2-preprocessing-outputs/<SimulationName>`. The name is built from the settings: `<lake>_<grid>_<start>-<end>[_ref<reference date>]_<weather model>[_<SimulationLabel>]`, e.g. `zug_100m_20250101-20250301_icon`.
All settings come from the `CurrentProject` block of `config.json`; data paths are resolved through `Config.Paths`.

**Steps**
1. Setup: load the config
2. Prepare the output folder from the templates of `templates-MITgcm/`
3. Grid and bathymetry
4. Surface forcings (Alplakes weather → `binary_data/*.bin`)
5. Initial conditions (Datalakes CTD profile, Simstrat profile or pickup)
6. Fill the template files (`run_config/data*`, `code/swfrac.F`, `postprocessing/*.py`)
7. Domain decomposition (`code/SIZE.h`, exch2)
8. Rivers (optional)
9. Launch script for kball_laken

**TODO:** choice of the pickup file, of the pickup and diagnostic frequencies, and of the output type (mnc or not).

## 1. Setup
The kernel must run from `preprocessing-mitgcm/` (plain imports of the library packages).

In [1]:
from lib.config import ConfigObject
from lib.grid_and_bathy import get_dz_grid, get_grid
from lib.initial_conditions import build_initial_conditions
from lib.rivers import build_river_config, build_river_binaries
from lib.run_folder import (prepare_output_folder, copy_grid_and_bathy, get_run_times, get_mitgcm_results_folder,
                               fill_run_config_templates, write_data_config_files, write_secchi,
                               write_size_config_files, write_exch2_config, write_launch_script,
                               write_watchdog_script, write_progress_script, write_fetch_results_script)
from lib.surface_forcings import download_weather, extract_and_save_surface_forcings

import os
import numpy as np

In [2]:
Config = ConfigObject('config.json')

# Start and end of the run, in seconds from the reference date (MITgcm time axis)
StartTimeInSecond, EndTimeInSecond = get_run_times(Config)

print(f'Simulation: {Config.SimulationName} ({Config.LakeName}, grid {Config.GridConfigName})')
print(f'Period: {Config.StartDate} -> {Config.EndDate} (reference {Config.ReferenceDate})')
print(f'Templates: {Config.TemplateFolder} | init: {Config.InitializationType} | rivers: {Config.WithRivers}')

Simulation: zug_100m_salt (zug, grid 100m)
Period: 20250101 -> 20250301 (reference 20250101)
Templates: ['default', 'exch2', 'obcs', 'crop_results'] | init: simstrat | rivers: False


## 2. Prepare the output folder
Wipes `2-preprocessing-outputs/<SimulationName>/` (other simulations are kept), overlays the template folders listed in `TemplateFolder` in order, and writes `metadata_simulation.txt`.

In [3]:
OutputFolder = prepare_output_folder(Config)
RunConfigFolder = os.path.join(OutputFolder, 'run_config')
BinaryDataFolder = os.path.join(OutputFolder, 'binary_data')
print(f'Output folder: {OutputFolder}')

Output folder: /c/Users/collinbr/Desktop/LZT_datafiles/2-preprocessing-outputs/zug_100m_salt


## 3. Grid and bathymetry
Grid and bathymetry are created beforehand with the notebooks of `grid_and_bathy/` (`create_grid`, `create_bathy`). They are copied to `grid/` and `binary_data/bathy.bin`.

In [4]:
copy_grid_and_bathy(Config, OutputFolder)

Grid = get_grid(Config.Paths.GridFolderPath)
DzGrid = get_dz_grid(os.path.join(Config.Paths.GridFolderPath, 'dz.csv'))
print(f'Grid: {Config.Nx} x {Config.Ny} cells, {np.count_nonzero(~np.isnan(DzGrid))} vertical layers')

Grid: 50 x 140 cells, 100 vertical layers


## 4. Surface forcings
Downloads the missing weather days from the Alplakes API into `1-preprocessing-inputs/raw_from_alplakes_api/<lake>` (days already there are skipped), then interpolates them to the grid and writes the forcing binaries (big-endian `>f8`) to `binary_data/`.

In [5]:
download_weather(Config, Grid)

Weather download (20250101-20250301): 0 downloaded, 480 already present, 0 failed


In [6]:
extract_and_save_surface_forcings(BinaryDataFolder, Config.StartDate, Config.EndDate, Config.Paths.RawWeatherFolder,
                                  Grid, Config.ALw, Config.WeatherModelType)

Finished: 100%|██████████ 9/9 [00:35]                                  

Surface forcings done in 36 s: 9 binary files written to /c/Users/collinbr/Desktop/LZT_datafiles/2-preprocessing-outputs/zug_100m_salt/binary_data
  period 20250101-20250301 (1417 hourly steps), grid 50 x 140 (X x Y), reanalysis data


## 5. Initial conditions
Depends on `InitializationType`:
- `datalakes_ctd` (default): temperature and salinity of the Canton Zug CTD profile (Datalakes, monthly at the deepest point) closest to the start date, downloaded into `1-preprocessing-inputs/datalakes_ctd/<lake>` if missing
- `simstrat`: Simstrat temperature profile closest to the start date (downloaded into `1-preprocessing-inputs/simstrat_data/<lake>` if missing), salinity from the CTD profile
- `pickup`: restart from a MITgcm pickup file; its number is the number of time steps since the reference date

Salinity evolves during the run (`saltStepping = .TRUE.`), with the same diffusivities and advection scheme as temperature.

In [7]:
InitialTemperature, InitialSalt, PickupNumber = build_initial_conditions(Config, DzGrid, StartTimeInSecond)

Simstrat data already exists: /c/Users/collinbr/Desktop/LZT_datafiles/1-preprocessing-inputs/simstrat_data/zug/simstrat_zug_202412300000_202501030000.json
Initial temperature from Simstrat profile of 2025-01-01T01:00:02.880000


## 6. Fill the template files
Replaces the `!placeholder!` tokens in the copied files only (never in `templates-MITgcm/`):
- `run_config/data.cal`, `data.exf`, `data.diagnostics`, `data.mnc` and `postprocessing/*.py`: dates, forcing parameters, results folders
- `code/swfrac.F`: Secchi depths (light penetration)
- `run_config/data`: run parameters, initial conditions, vertical grid

In [8]:
# Results folder on the computer running MITgcm
MitgcmResultsFolder = get_mitgcm_results_folder(Config)

fill_run_config_templates(Config, OutputFolder, MitgcmResultsFolder)
write_secchi(os.path.join(OutputFolder, 'code/swfrac.F'), Config.Secchi)
write_data_config_files(os.path.join(RunConfigFolder, 'data'), InitialTemperature, InitialSalt,
                        StartTimeInSecond, EndTimeInSecond, PickupNumber, DzGrid, Config)
print(f'Results folder on {Config.ComputerName}: {MitgcmResultsFolder}')

Results folder on kball_laken: ../../../3-MITgcm-outputs/zug_100m_salt


## 7. Domain decomposition
With the `exch2` template, the tiles that contain only land are dropped: the land tiles are read from `land_cores_Px{Px}_Py{Py}.txt` (written by `grid_and_bathy/check_core_allocation.ipynb`), and the remaining cores are laid out as a single row (`Py = 1`).

In [9]:
UseExch2 = 'exch2' in Config.TemplateFolder
Px, Py = write_exch2_config(OutputFolder, Config) if UseExch2 else (Config.Px, Config.Py)

# Tile size (sNx, sNy) comes from the full decomposition of config.json, before land tiles are dropped
write_size_config_files(os.path.join(OutputFolder, 'code/SIZE.h'), Px, Py, Config.Nx, Config.Ny,
                        np.count_nonzero(~np.isnan(DzGrid)), int(Config.Nx / Config.Px), int(Config.Ny / Config.Py))
print(f'Using exch2: {UseExch2} | number of cores: {Px * Py}')

Using exch2: True | number of cores: 33


## 8. Rivers (optional)
Rivers enter as open boundaries (only if `WithRivers` is true).

In [10]:
if Config.WithRivers:
    RiverDicts, RiverString = build_river_config(Config, RunConfigFolder)
    build_river_binaries(Config, BinaryDataFolder)
    print(RiverString)
else:
    print('No rivers')

No rivers


## 9. Launch script
Writes `launch_simulation.sh`, which compiles and starts the simulation on the remote computer in one command (see "Next steps"), and `watchdog.sh`, which the launch script starts next to MITgcm. Every `CheckInterval` seconds the watchdog reads the statistics MITgcm writes to `run/STDOUT.0000` every `MonitorFreq` model seconds; on a NaN or a value outside `Limits` it stops the run. It sends an email (`mail` command on kball, to `NotifyEmail`) when the run finishes, fails or is stopped. Settings: `config.json`, `ComputerConfig.Watchdog`. Finally `progress.sh`, the progress bar the launch script shows until the end of the run. Last, `fetch_results.sh`, run on kball by `Fetch results from kball_laken.bat` to download the results only if the run finished without errors.

In [11]:
LaunchScriptPath = write_launch_script(OutputFolder, Config, Px * Py, MitgcmResultsFolder)
print(f'Written: {LaunchScriptPath}')

# Watchdog started by the launch script: stops the run on unrealistic values, emails when the run ends
WatchdogPath = write_watchdog_script(OutputFolder, Config, MitgcmResultsFolder)
print(f'Written: {WatchdogPath} (email: {Config.Watchdog["NotifyEmail"] or "none, set NotifyEmail in config.json"})')

# Progress bar shown by the launch script (and on demand with 'bash <simulation folder>/progress.sh')
ProgressPath = write_progress_script(OutputFolder, Config, MitgcmResultsFolder, StartTimeInSecond, EndTimeInSecond)
print(f'Written: {ProgressPath}')

# Run on kball by 'Fetch results from kball_laken.bat': sends the results only if the run finished without errors
FetchResultsPath = write_fetch_results_script(OutputFolder, Config, MitgcmResultsFolder)
print(f'Written: {FetchResultsPath}')

Written: /c/Users/collinbr/Desktop/LZT_datafiles/2-preprocessing-outputs/zug_100m_salt/launch_simulation.sh


## Next steps

The simulation folder `LZT_datafiles/2-preprocessing-outputs/<SimulationName>` is ready. On kball_laken, the data folders sit in `/storage/lake_zug_transport` (`ComputerConfig.RemoteRoot`), with the same layout as `LZT_datafiles`:

```
/storage/lake_zug_transport/2-preprocessing-outputs/<SimulationName>/   simulation folder (copied in step 1)
/storage/lake_zug_transport/3-MITgcm-outputs/<SimulationName>/          MITgcm results (created by the launch script)
~/MITgcm-checkpoint67c/                                                 MITgcm source (ComputerConfig.MitgcmRootDir)
```

**1. Send the simulation folder to kball** (on the local Windows machine)

Double-click `Send Preprocess to kball_laken.bat` on the Desktop, type the number of the simulation in the list (the folders of `LZT_datafiles/2-preprocessing-outputs`), then your password. scp shows the progress of each file, and the simulation folder lands in `/storage/lake_zug_transport/2-preprocessing-outputs/` (existing files are overwritten).

**2. Start the simulation** (on kball, accessible through the `VScode_LZT kball_laken.bat` on the Desktop), in a terminal opened in any folder)

```bash
bash /storage/lake_zug_transport/2-preprocessing-outputs/<SimulationName>/launch_simulation.sh
```

The script compiles MITgcm (`genmake2`, `make`) in `build/`, prepares `run/`, creates `/storage/lake_zug_transport/3-MITgcm-outputs/<SimulationName>/` (linked as `run/results`) and starts MITgcm with `mpirun` in the background. Everything it prints until MITgcm starts is also saved in `launch.log`, and if a step fails (e.g. the compilation) the end of that log is emailed to `NotifyEmail` (`config.json`, `ComputerConfig.Watchdog`). It also starts `watchdog.sh`, which emails right away ("started": if this email does not arrive, emails do not work from kball, see `run/watchdog.log`), stops the run if the model statistics become NaN or unrealistic, and emails again when the run finishes, fails or is stopped. It then shows a progress bar (model days done, time left) until the end of the run; Ctrl+C or closing the terminal only stops the display, MITgcm and the watchdog keep running. The MITgcm folder and build options come from `config.json` (`ComputerConfig`); another MITgcm folder can be used with `MITGCM_ROOTDIR=/path/to/MITgcm bash /storage/lake_zug_transport/2-preprocessing-outputs/<SimulationName>/launch_simulation.sh`.

**3. Follow the run** (on kball; the launch script also prints these commands with the full paths)

```bash
cd /storage/lake_zug_transport/2-preprocessing-outputs/<SimulationName>
tail -f run/STDOUT.0000                  # MITgcm log (first core)
cat run/STDERR.0000                      # MITgcm errors
cat run/output_<SimulationName>.out      # mpirun messages (crash)
grep 'NORMAL END' run/STDOUT.0000        # run finished properly?
ps aux | grep mpirun                     # running simulations
cat run/watchdog.log                     # watchdog messages
bash progress.sh                         # progress bar again (Ctrl+C stops the display only)
```

Email check (once, on kball): `echo test | mail -s test <NotifyEmail>`. Without a working `mail`, the watchdog only writes to `run/watchdog.log`, which also says, for each email, whether `mail` accepted it.

**4. Results**

- `/storage/lake_zug_transport/3-MITgcm-outputs/<SimulationName>/outputs/`: hourly snapshots `3Dsnaps.*.data/.meta` (diagnostics). These are the results to analyse.
- `/storage/lake_zug_transport/3-MITgcm-outputs/<SimulationName>/to_delete/`: NetCDF monitor output, can be deleted.
- `/storage/lake_zug_transport/run/`: logs (`STDOUT.*`), grid and pickup files.

**5. Download the results** (on the local Windows machine, after the watchdog's "finished" email, which reminds it)

Double-click `Fetch results from kball_laken.bat` on the Desktop, type the number of the simulation in the list (the folders of `LZT_datafiles/2-preprocessing-outputs`, already downloaded ones are tagged), then your password. Through the same ssh connection, kball first checks that the run finished without errors (not running anymore, `NORMAL END` in `run/STDOUT.0000`). If the run failed, the reason and the end of the logs are shown and the `.bat` asks whether to download the results and logs anyway (password asked again). Then kball copies the logs of the run to `logs/` and sends `/storage/lake_zug_transport/3-MITgcm-outputs/<SimulationName>/` with a progress bar (%, GB, speed, time left), extracted to `LZT_datafiles/3-MITgcm-outputs/<SimulationName>/` (about 16 GB for 2 months; this folder is read-only for the code). If that folder already exists, you choose: cancel, download everything again (the old results are replaced only once the new download is complete), or download only the files not downloaded yet.